# Creating semantic search from dataset
Transformer-based language models represent each token in a span of text as an embedding vector.
When creating embedings model create whole representation of the text as a vector space.
When we provide query and pass it through same model, we can compare query embedding with vectors space and retrieve most similar documents.

Note:
Embedings it's just a numerical representation of text.

![image](assets/vector_space.png)

## Get dataset

In [ ]:
from datasets import load_dataset

issues_dataset = load_dataset("lewtun/github-issues", split="train")
issues_dataset

Preprocessing dataset to get rid of noise and make it more suitable for semantic search.

In [ ]:
# Remove pull requests and issues without comments
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

In [ ]:
# Leave only necessary columns for semantic search
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

In [ ]:
# Explode comments to have one comment per row
issues_dataset.set_format("pandas")
df = issues_dataset[:]

In [ ]:
df.head()

In [ ]:
df["comments"][0].tolist()

In [ ]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)

In [ ]:
# Convert to Hugging Face Dataset again
from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

In [ ]:
# Create comment length column and filter out comments that are too short
comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

In [ ]:
# Create single column with title, body and comment for semantic search
def concatenate_text(examples):
    return {
        "text": examples["title"]
        + " \n "
        + examples["body"]
        + " \n "
        + examples["comments"]
    }
comments_dataset = comments_dataset.map(concatenate_text)

In [ ]:
comments_dataset['text'][0]

## Creating text embeddings

- asymmetric semantic search - when we try from short query to find longer documents
- we need to use HF AutoModel to get embeddings.

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

We need to average our tokens in each row to get a vector representation of the whole sequence. Pooling is a technique to do that.
It averages the single token embeddings to get a single vector represantation of the whole sequence.

In [ ]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

In [ ]:
import torch

device = torch.device("cpu")
model.to(device)

In [ ]:
def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

In [ ]:
# Embeddings for the first row of our dataset
embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

In [ ]:
# And for a whole dataset
embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

In [ ]:
embeddings_dataset

In [ ]:
embeddings_dataset['embeddings'][0][0:3]

In [ ]:
embeddings_dataset.save_to_disk("embeddings_dataset")

In [ ]:
from datasets import load_from_disk

In [ ]:
embeddings_dataset = load_from_disk("embeddings_dataset")

In [ ]:
embeddings_dataset

## Semantic search FAISS
![image](assets/faiss.png)

Facebook AI Similarity Search

Formula above simple means:
 - We take some set of numbers (document embedding in our case) e.x. [5, 10, 15]
 - We have our intrest number (query embedding) e.x. [7]
 - We need to find nearest point using euclidiean distance formula sqrt((7-5)^2) = 2, sqrt((7-10)^2) = 3, sqrt((7-15)^2) = 8
 - argmin part says that we need to take smallest distance from the calucaltions above in our case 2.
 - That means that 5 is the closest point to 7 in our set of numbers.


With FAISS we need to create index for a search.

In [ ]:
!pip install faiss-cpu

In [ ]:
embeddings_dataset.add_faiss_index(column="embeddings")

Now we can embed our query.

In [ ]:
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

We get same shape as before so everything seems to be correct.

Now we apply nearest neighbor search to find the most similar embeddings to our query embedding.

In [ ]:
scores, samples = embeddings_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

In [ ]:
scores

In [ ]:
samples

In [ ]:
import pandas as pd

samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

In [ ]:
samples_df

In [ ]:
print(f"QUESTION: {question}")
print("=" * 50)
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

Seems that second comment is the most similar to our query.